# Checkpoint 5C — Multi-Satellite Magnetic Generality

**Question:** Does the descriptive low-`Btot_sat` framing found for NOAA-19 remain broadly consistent across the five January-2024 satellites accepted in CP4F?

Every satellite defines its high-flux footprint from its own within-satellite mean-flux percentile distribution. Absolute proton flux is not compared across satellites. The frozen rubric below is an operational CP5C decision rule, not a physical threshold defining the SAA. Results are descriptive, January-only, method-dependent, non-causal, and not a final boundary.

In [1]:
import hashlib, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from saa.magnetic_audit import load_region_with_magnetic
from saa.magnetic_generality import (
    SATELLITES, ReferenceBundle, add_cp5c_footprint_flags, compare_noaa19_reference,
    finalize_generality_summary, footprint_cell_sets, ifc_counts, omni_fit_flag_diagnostic,
    plot_capture90_comparison, plot_separation_comparison, principal_summary_row,
    summaries_by_satellite, validity_by_satellite,
)

RAW = ROOT / 'data' / 'raw'; PROC = ROOT / 'data' / 'processed'
TBL = ROOT / 'outputs' / 'tables'; FIG = ROOT / 'outputs' / 'figures'
for directory in (RAW, PROC, TBL, FIG): directory.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 240, 'display.max_columns', 40)
print('CP5C fixed scope:', SATELLITES, '| January 2024 | p1 | top10/top5 | 5/2 degree mean')

CP5C fixed scope: ('noaa15', 'noaa18', 'noaa19', 'metop01', 'metop03') | January 2024 | p1 | top10/top5 | 5/2 degree mean


## 1. Accepted-artifact integrity and one-satellite processing

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''): digest.update(block)
    return digest.hexdigest()

accepted_artifacts = sorted(
    [p for prefix in ('cp4f_', 'cp5a_', 'cp5b_') for p in TBL.glob(prefix + '*')]
    + [p for prefix in ('cp4f_', 'cp5a_') for p in PROC.glob(prefix + '*')]
)
if not accepted_artifacts:
    raise FileNotFoundError('accepted CP4F/CP5A/CP5B artifacts are absent; regenerate prerequisites first')
accepted_hashes_before = {str(path.relative_to(ROOT)): sha256(path) for path in accepted_artifacts}

def save_pair(frame, stem):
    frame.to_csv(TBL / f'{stem}.csv', index=False)
    frame.to_parquet(TBL / f'{stem}.parquet', index=False)

def process_satellite(satellite):
    region, preparation, paths = load_region_with_magnetic(
        '2024-01-01', '2024-01-31', satellite=satellite, raw_dir=str(RAW)
    )
    if len(paths) != 31:
        raise RuntimeError(f'{satellite}: expected 31 real NOAA files, got {len(paths)}')
    region.to_parquet(PROC / f'cp5c_{satellite}_2024-01_region_flux_plus_magnetic.parquet', index=False)
    grid5 = pd.read_parquet(TBL / f'cp4f_{satellite}_2024-01_grid_5deg.parquet')
    grid2 = pd.read_parquet(TBL / f'cp4f_{satellite}_2024-01_grid_2deg.parquet')
    cells = footprint_cell_sets(grid5, grid2)
    flagged = add_cp5c_footprint_flags(region, grid5, grid2)
    validity = validity_by_satellite(flagged, satellite)
    summary, concentration = summaries_by_satellite(flagged, satellite)
    processing = ifc_counts(region, preparation)
    processing['top10_5deg_selected_cell_count'] = len(cells['top10_5deg_mean'])
    principal = principal_summary_row(
        satellite, summary, concentration, validity, flagged, processing
    )
    fit_flags = omni_fit_flag_diagnostic(flagged, satellite)
    return {
        'region': region, 'flagged': flagged, 'validity': validity, 'summary': summary,
        'concentration': concentration, 'fit_flags': fit_flags, 'principal': principal,
        'cell_sets': cells, 'grid5': grid5, 'grid2': grid2,
    }

print('Accepted artifact integrity snapshot:', len(accepted_hashes_before), 'files')

Accepted artifact integrity snapshot: 34 files


## 2. NOAA-19 hard reproducibility gate

Exact comparisons cover selected cell sets, counts, and expected case/variable/metric keys. Floating comparisons use the predeclared `rtol=1e-9`, `atol=1e-12`, `equal_nan=True`; these tolerances are not adjusted after execution.

In [3]:
results = {}
results['noaa19'] = process_satellite('noaa19')
actual_reference = ReferenceBundle(
    cell_sets=results['noaa19']['cell_sets'],
    validity=results['noaa19']['validity'],
    footprint_summary=results['noaa19']['summary'],
    concentration=results['noaa19']['concentration'],
)
cp4a_grid5 = pd.read_parquet(TBL / 'cp4a_noaa19_2024-01_grid_5deg.parquet')
cp4a_grid2 = pd.read_parquet(TBL / 'cp4a_noaa19_2024-01_grid_2deg.parquet')
expected_reference = ReferenceBundle(
    cell_sets=footprint_cell_sets(cp4a_grid5, cp4a_grid2),
    validity=pd.read_csv(TBL / 'cp5b_magnetic_variable_validity.csv'),
    footprint_summary=pd.read_parquet(TBL / 'cp5b_footprint_magnetic_summary.parquet'),
    concentration=pd.read_csv(TBL / 'cp5b_magnetic_concentration_metrics.csv'),
)
compare_noaa19_reference(actual_reference, expected_reference)
print('NOAA-19 HARD GATE: PASS (exact discrete comparisons; rtol=1e-9, atol=1e-12, equal_nan=True)')

NOAA-19 HARD GATE: PASS (exact discrete comparisons; rtol=1e-9, atol=1e-12, equal_nan=True)


## 3. Process the remaining CP4F satellites after the gate

In [4]:
for satellite in SATELLITES:
    if satellite != 'noaa19':
        results[satellite] = process_satellite(satellite)

validity = pd.concat([results[s]['validity'] for s in SATELLITES], ignore_index=True)
summary = pd.concat([results[s]['summary'] for s in SATELLITES], ignore_index=True)
concentration = pd.concat([results[s]['concentration'] for s in SATELLITES], ignore_index=True)
fit_flags = pd.concat([results[s]['fit_flags'] for s in SATELLITES], ignore_index=True)
generality = finalize_generality_summary([results[s]['principal'] for s in SATELLITES])
save_pair(validity, 'cp5c_magnetic_variable_validity_by_satellite')
save_pair(summary, 'cp5c_footprint_magnetic_summary_by_satellite')
save_pair(concentration, 'cp5c_magnetic_concentration_by_satellite')
save_pair(generality, 'cp5c_multisatellite_magnetic_generality_summary')
save_pair(fit_flags, 'cp5c_omni_fit_flag_diagnostic')
print('Saved CP5C tables:', len(validity), len(summary), len(concentration), len(generality), len(fit_flags))

Saved CP5C tables: 25 80 80 5 26


## 4. Frozen rubric result and raw satellite-level evidence

In [5]:
rubric_columns = [
    'satellite', 'btot_separation_metric', 'l_igrf_separation_metric', 'mlt_separation_metric',
    'btot_fraction_below_regional_q25', 'btot_regional_fraction_to_capture_50pct',
    'btot_regional_fraction_to_capture_75pct', 'btot_regional_fraction_to_capture_90pct',
    'low_btot_support', 'btot_dominance_support',
]
classification = generality['cp5c_classification'].iloc[0]
print('PREDECLARED CP5C RUBRIC RESULT:', classification)
print('Operational criteria only; not physical thresholds defining the SAA.\n')
print(generality[rubric_columns].to_string(index=False))
print('\nSupport counts:', int(generality['low_btot_support_count'].iloc[0]), 'low-Btot;',
      int(generality['btot_dominance_support_count'].iloc[0]), 'Btot-dominance;',
      int(generality['reversed_btot_sign_count'].iloc[0]), 'negative Btot signs')

PREDECLARED CP5C RUBRIC RESULT: CONSISTENT
Operational criteria only; not physical thresholds defining the SAA.

satellite  btot_separation_metric  l_igrf_separation_metric  mlt_separation_metric  btot_fraction_below_regional_q25  btot_regional_fraction_to_capture_50pct  btot_regional_fraction_to_capture_75pct  btot_regional_fraction_to_capture_90pct  low_btot_support  btot_dominance_support
   noaa15                1.596147                  0.387097               0.919967                               1.0                                 0.051802                                 0.083628                                 0.118963              True                    True
   noaa18                1.572744                  0.428571               0.048053                               1.0                                 0.052012                                 0.086146                                 0.120877              True                    True
   noaa19                1.584260        

## 5. Compact descriptive figures and preservation check

In [6]:
plot_separation_comparison(
    generality, FIG / 'cp5c_multisatellite_magnetic_separation_top10_5deg_mean.png'
)
plot_capture90_comparison(
    generality, FIG / 'cp5c_multisatellite_low_btot_capture90_top10_5deg_mean.png'
)
accepted_hashes_after = {str(path.relative_to(ROOT)): sha256(path) for path in accepted_artifacts}
if accepted_hashes_after != accepted_hashes_before:
    changed = sorted(set(accepted_hashes_before) | set(accepted_hashes_after))
    changed = [name for name in changed if accepted_hashes_before.get(name) != accepted_hashes_after.get(name)]
    raise AssertionError(f'accepted artifacts changed during CP5C: {changed}')
print('Figures written; accepted CP4F/CP5A/CP5B artifacts unchanged:', len(accepted_hashes_after), 'hashes match')

Figures written; accepted CP4F/CP5A/CP5B artifacts unchanged: 34 hashes match


## 6. Narrative interpretation after the rubric result

In [7]:
if classification == 'CONSISTENT':
    narrative = 'The low-Btot descriptive relationship and Btot dominance are broadly consistent across the tested satellites under the predeclared CP5C criteria.'
elif classification == 'MIXED':
    narrative = 'The tested satellites provide mixed evidence under the predeclared CP5C criteria; the satellite-level metrics above show which part of the framing weakens or reorders.'
else:
    narrative = 'The NOAA-19 low-Btot framing is inconsistent across the tested satellites under the predeclared CP5C criteria.'
print(narrative)
print('This is within-satellite, descriptive January-2024 evidence: not causal, not a universal Btot threshold, not temporal generality, and not a final SAA boundary.')

The low-Btot descriptive relationship and Btot dominance are broadly consistent across the tested satellites under the predeclared CP5C criteria.
This is within-satellite, descriptive January-2024 evidence: not causal, not a universal Btot threshold, not temporal generality, and not a final SAA boundary.
